# AlgoChowk — NIFTY Event Study & Simple Backtest

Hypothesis: after a significant one-day fall in NIFTY 50, the index tends to recover over the next few trading days.

**Default experiment:** event threshold = -2%, holding periods = 3, 5, 10 trading days. Entry is next day's Open to avoid look-ahead bias. Transaction cost/slippage is configurable.


In [ ]:
!pip -q install yfinance scipy statsmodels


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
import matplotlib.pyplot as plt

THRESHOLD = -0.02       # significant one-day fall
HOLDING_PERIODS = [3, 5, 10]
COST_PER_TRADE = 0.0005 # 5 bps total assumed trading friction
START = '2010-01-01'
END = None

# NIFTY 50 Yahoo Finance symbol
raw = yf.download('^NSEI', start=START, end=END, auto_adjust=False, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw.reset_index()
raw.columns = [c.title() for c in raw.columns]
raw = raw[['Date','Open','High','Low','Close']].copy()
raw['Date'] = pd.to_datetime(raw['Date'])
raw = raw.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)

# Validation
print('Rows:', len(raw))
print('Date range:', raw.Date.min().date(), 'to', raw.Date.max().date())
print('Missing values:\n', raw.isna().sum())
print('Duplicate dates:', raw.Date.duplicated().sum())
assert raw[['Open','High','Low','Close']].notna().all().all()
assert (raw[['Open','High','Low','Close']] > 0).all().all()

raw['event_return'] = raw['Close'].pct_change()


In [ ]:
# Detect events and calculate forward returns.
# Event is known after the event day's close; therefore entry = next trading day's Open.

events = raw.loc[raw['event_return'] <= THRESHOLD].copy()
events['event_date'] = events['Date']
events['entry_date'] = raw['Date'].shift(-1).loc[events.index].values
events['entry_price'] = raw['Open'].shift(-1).loc[events.index].values

for h in HOLDING_PERIODS:
    # Exit at the close h trading days after entry day.
    exit_idx = events.index + h
    valid = exit_idx < len(raw)
    vals = np.full(len(events), np.nan)
    vals[valid] = raw['Close'].iloc[exit_idx[valid]].to_numpy()
    events[f'{h}d_return'] = vals / events['entry_price'].to_numpy() - 1 - COST_PER_TRADE

events = events.dropna(subset=['entry_price']).reset_index(drop=True)
print('Number of qualifying events:', len(events))
events[['event_date','event_return','entry_date','entry_price'] + [f'{h}d_return' for h in HOLDING_PERIODS]].head(10)


In [ ]:
def summary(series):
    s = series.dropna()
    return pd.Series({
        'N': len(s),
        'Mean': s.mean(),
        'Median': s.median(),
        'Win rate': (s > 0).mean(),
        'Std dev': s.std(ddof=1),
    })

results = pd.DataFrame({f'{h}d': summary(events[f'{h}d_return']) for h in HOLDING_PERIODS}).T
results


In [ ]:
# Baseline: all normal next-h-day close-to-close returns over the same sample period.
baseline = {}
for h in HOLDING_PERIODS:
    r = raw['Close'].shift(-h) / raw['Close'] - 1 - COST_PER_TRADE
    baseline[f'{h}d'] = summary(r)
baseline = pd.DataFrame(baseline).T
comparison = results[['N','Mean','Median','Win rate']].copy()
comparison['Baseline Mean'] = baseline['Mean']
comparison['Mean Difference'] = comparison['Mean'] - baseline['Mean']
comparison


In [ ]:
# Simple statistical evidence: one-sample t-test of event returns vs 0.
tests = []
for h in HOLDING_PERIODS:
    s = events[f'{h}d_return'].dropna()
    t, p = stats.ttest_1samp(s, 0) if len(s) > 1 else (np.nan, np.nan)
    tests.append([h, t, p])
pd.DataFrame(tests, columns=['Holding','t_stat','p_value'])


In [ ]:
# Out-of-sample split: first 70% development, last 30% unseen.
split_date = raw['Date'].iloc[int(len(raw)*0.70)]
dev = events[events['event_date'] < split_date]
oos = events[events['event_date'] >= split_date]
print('Split:', split_date.date())
print('Development events:', len(dev))
print('Out-of-sample events:', len(oos))

oos_summary = pd.DataFrame({f'{h}d': summary(oos[f'{h}d_return']) for h in HOLDING_PERIODS}).T
oos_summary


In [ ]:
# Robustness: vary event threshold without changing the research engine.
def run_threshold(threshold, holding=5):
    idx = raw.index[raw['event_return'] <= threshold].to_numpy()
    vals = []
    for i in idx:
        entry_i = i + 1
        exit_i = i + 1 + holding
        if exit_i < len(raw):
            vals.append(raw.loc[exit_i,'Close'] / raw.loc[entry_i,'Open'] - 1 - COST_PER_TRADE)
    return summary(pd.Series(vals))

robust = pd.DataFrame({f'{x:.0%}': run_threshold(x, 5) for x in [-0.015,-0.02,-0.025,-0.03]}).T
robust


In [ ]:
# Event-driven 5-day backtest. One position per qualifying event; overlapping events are skipped.
signal = raw['event_return'] <= THRESHOLD
trades = []
next_free = 0
for i in np.where(signal)[0]:
    entry_i, exit_i = i+1, i+1+5
    if entry_i >= len(raw) or exit_i >= len(raw) or entry_i < next_free:
        continue
    entry = raw.loc[entry_i,'Open']
    exitp = raw.loc[exit_i,'Close']
    ret = exitp/entry - 1 - COST_PER_TRADE
    trades.append([raw.loc[i,'Date'], raw.loc[entry_i,'Date'], raw.loc[exit_i,'Date'], entry, exitp, ret])
    next_free = exit_i + 1

trades = pd.DataFrame(trades, columns=['Event Date','Entry Date','Exit Date','Entry','Exit','Return'])
trades['Equity'] = (1 + trades['Return']).cumprod() if len(trades) else []
trades


In [ ]:
if len(trades):
    equity = trades['Equity']
    peak = equity.cummax()
    drawdown = equity/peak - 1
    print('Trades:', len(trades))
    print('Cumulative return:', equity.iloc[-1]-1)
    print('Maximum drawdown:', drawdown.min())
    equity.plot(title='5-Day Event-Driven Backtest Equity Curve')
    plt.ylabel('Growth of ₹1')
    plt.show()
else:
    print('No valid trades under the selected parameters.')


## Interpretation checklist

- Do not claim recovery merely because the event mean is positive.
- Compare event returns with the baseline.
- Check whether results persist out-of-sample.
- Treat threshold/holding-period changes as robustness checks, not as a search for the best result.
- Discuss look-ahead bias, transaction costs, slippage, overlapping events, sample size, market regimes and data quality.
- A negative result is acceptable; the goal is evidence, not a good backtest.
